Code for checking, validating, and collecting critical MEG acquisition parameters.

**Analogous to the 'collect_fMRI_parameters' script from the fMRI pipeline.** Caveat: unlike in the case of fMRI, we shouldn't need to directly read this file to provide parameters during MEG processing; rather, this is more just for data auditing/screening/QC.

**Summary:**
This script takes the 'master data catalogue.csv', created in the preceding scripts, and expands it into a new run-level MEG index table: one row per MEG recording. It then opens each MEG '.fif' file just enough to read its header, and extracts key acquisition metadata (sampling rate, duration, channel composition, line frequency, device/HPI fields). The main purpose here is to catch “hidden incompatibilities” early: like mismatched sampling frequencies; missing HPI information; or weird channel layouts, *before* committing to running preprocessing/source modeling on data that will later fail or silently misbehave.

[Runtime: Dataset size-dependent, but most likely under 2 minutes in total]

---------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path
import subprocess

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os, json, re, glob
from pathlib import Path
import numpy as np
import pandas as pd
import mne
from mne.io.constants import FIFF

In [ ]:
### SET PARAMETERS, AND INPUT & OUTPUT FILEPATHS:

### PARAMETERS:
HARD_STOP = config['hard_errors']
EXPORT_WARNING_CSV = config['export_MEG_metadata_warnings']

DROP_MISSING_HPI = config['drop_missing_HPI_info']
FORCE_SAME_SAMPLING_FREQUENCY = config['force_same_sampling_freq']
DISCARD_SFREQ_MISMATCHED = config['discard_sfreq_mismatch']

### INPUTS:
ROOT_DIR = Path(config['root_output_directory'])
MEG_DATA_DIR = Path(config['MEG_data_directory'])
DATA_INDEX_PATH = Path(ROOT_DIR) / 'master_data_catalogue.csv'
DATA_INDEX = pd.read_csv(DATA_INDEX_PATH)

### OUTPUTS:
PARAMETER_INDEX_FILEPATH = Path(ROOT_DIR) / 'MEG_parameter_index.csv'

print(f"Main project directory set as: {ROOT_DIR}")
print(f"Main MRI input data directory set as: {MEG_DATA_DIR}")
print(f"Master data catalogue set as: {DATA_INDEX_PATH}")
print()
print(f"MEG parameter catalogue will be saved to: {PARAMETER_INDEX_FILEPATH}")

First, let's build a base index of files to grab parameters for, by taking all non-NaN values from our various 'filename' columns, along with subject-, group-, and session IDs:

In [ ]:
# =========================
# Build base MEG file index from master data catalogue table:
# =========================

# Identify all columns containing both 'MEG' and 'filename':
MEG_filename_cols = [c for c in DATA_INDEX.columns if ('MEG' in c) and ('filename' in c)]

records = []
for _, row in DATA_INDEX.iterrows():
    subject_id = row.get('subject_ID', np.nan)
    group_id   = row.get('group_ID',   np.nan)
    for col in MEG_filename_cols:
        val = row[col]
        # keep only non-empty values:
        if pd.notna(val) and str(val).strip():
            # Derive session_ID from the column name: 'MEG_<session>_filename':
            parts = col.split('_')
            session_id = parts[1] if len(parts) >= 3 else np.nan
            records.append({
                'subject_ID':     subject_id,
                'group_ID':       group_id,
                'session_ID':     session_id,
                'MEG_filename':  str(val).strip()})  # base path (no extension)

# Assemble the dataframe:
index = pd.DataFrame.from_records(
    records,
    columns=['subject_ID', 'group_ID', 'session_ID', 'MEG_filename'])

# Ensure uniqueness by filename (keep first occurrence):
if not index.empty:
    before = len(index)
    index = index.drop_duplicates(subset=['MEG_filename']).reset_index(drop=True)
    removed = before - len(index)
else:
    removed = 0

print(f"[INFO] Scanned {len(MEG_filename_cols)} MEG filename columns.")
print(f"[INFO] Collected {len(records)} total filename hits.")
print(f"[INFO] Removed {removed} duplicate filename rows.")
print(f"[OK] Built base index with {len(index)} unique MEG files.")

Next, open FIF files & collect metadata & acquisition parameters:

In [ ]:
# =========================
# MEG header extraction from FIF
# Adds: MEG_fullpath, sfreq_Hz, n_times, duration_sec, channel counts,
#       filter settings, line frequency, device info, HPI info
# =========================

### Initialize 'problems_df' to log all data-auditing problems:
problems_df = pd.DataFrame(
    columns=["reason", "subject_ID", "group_ID", "session_ID", "MEG_fullpath"])

# New columns to hold per-run MEG metadata
index["MEG_fullpath"]        = np.nan

index["sfreq_Hz"]            = np.nan
index["n_times"]             = np.nan
index["duration_sec"]        = np.nan
index["duration_min"]        = np.nan

index["n_channels_total"]    = np.nan
index["n_meg_channels"]      = np.nan
index["n_ref_channels"]      = np.nan
index["n_eeg_channels"]      = np.nan
index["n_stim_channels"]     = np.nan
index["n_misc_channels"]     = np.nan
index["n_other_channels"]    = np.nan  # anything not in the above types

index["highpass_Hz"]         = np.nan
index["lowpass_Hz"]          = np.nan
index["line_freq_Hz"]        = np.nan

index["device_manufacturer"] = np.nan
index["device_model"]        = np.nan
index["device_type"]         = np.nan

index["n_hpi_meas"]          = np.nan
index["n_hpi_results"]       = np.nan
index["n_hpi_coils"]         = np.nan
index["has_HPI"]             = np.nan  # boolean flag

errors = []

for i, row in index.iterrows():
    base_str = str(row["MEG_filename"]).strip()
    if base_str.endswith(".fif"):
        base_str = base_str[:-len(".fif")]
    if not base_str:
        continue

    # We assume a flat FIF layout: <MEG_DATA_DIR>/<basename>.fif
    meg_path = MEG_DATA_DIR / f"{base_str}.fif"

    if not meg_path.exists():
        errors.append((
            row.get("subject_ID", np.nan),
            row.get("group_ID",   np.nan),
            row.get("session_ID", np.nan),
            str(meg_path),
            "missing MEG FIF file"))
        continue

    try:
        raw = mne.io.read_raw_fif(meg_path, preload=False, verbose="ERROR")

        # Basic timing info
        sfreq = float(raw.info["sfreq"])
        n_times = int(raw.n_times)
        duration_sec = n_times / sfreq if sfreq > 0 else np.nan
        duration_min = duration_sec / 60.0 if sfreq > 0 else np.nan

        # Channel types & counts
        ch_types = raw.get_channel_types()
        n_total = len(ch_types)
        n_meg  = sum(t in ("mag", "grad", "meg") for t in ch_types)
        n_eeg  = ch_types.count("eeg")
        n_stim = ch_types.count("stim")
        n_misc = ch_types.count("misc")

        # Reference MEG channels (if present in this system):
        picks_ref = mne.pick_types(raw.info, ref_meg=True, meg=False, eeg=False, stim=False, misc=False)
        n_ref = int(len(picks_ref))

        n_other = n_total - (n_meg + n_ref + n_eeg + n_stim + n_misc)

        # Filter / line freq
        highpass = raw.info.get("highpass", np.nan)
        lowpass  = raw.info.get("lowpass", np.nan)
        line_freq = raw.info.get("line_freq", np.nan)

        # Device info (if present)
        device_info = raw.info.get("device_info") or {}
        manuf = device_info.get("manufacturer", np.nan)
        model = device_info.get("model", np.nan)
        dtype = device_info.get("type", np.nan)

        # HPI info
        hpi_meas    = raw.info.get("hpi_meas") or []
        hpi_results = raw.info.get("hpi_results") or []
        dig         = raw.info.get("dig") or []

        n_hpi_meas    = len(hpi_meas)
        n_hpi_results = len(hpi_results)
        n_hpi_coils   = sum(
            1 for d in dig
            if isinstance(d, dict) and d.get("kind") == FIFF.FIFFV_POINT_HPI)
        has_hpi = bool((n_hpi_meas > 0) or (n_hpi_results > 0) or (n_hpi_coils > 0))

        # Store into dataframe
        index.at[i, "MEG_fullpath"]        = meg_path.resolve().as_posix()

        index.at[i, "sfreq_Hz"]            = sfreq
        index.at[i, "n_times"]             = n_times
        index.at[i, "duration_sec"]        = duration_sec
        index.at[i, "duration_min"]        = duration_min

        index.at[i, "n_channels_total"]    = n_total
        index.at[i, "n_meg_channels"]      = n_meg
        index.at[i, "n_ref_channels"]      = n_ref
        index.at[i, "n_eeg_channels"]      = n_eeg
        index.at[i, "n_stim_channels"]     = n_stim
        index.at[i, "n_misc_channels"]     = n_misc
        index.at[i, "n_other_channels"]    = n_other

        index.at[i, "highpass_Hz"]         = highpass
        index.at[i, "lowpass_Hz"]          = lowpass
        index.at[i, "line_freq_Hz"]        = line_freq

        index.at[i, "device_manufacturer"] = manuf
        index.at[i, "device_model"]        = model
        index.at[i, "device_type"]         = dtype

        index.at[i, "n_hpi_meas"]          = n_hpi_meas
        index.at[i, "n_hpi_results"]       = n_hpi_results
        index.at[i, "n_hpi_coils"]         = n_hpi_coils
        index.at[i, "has_HPI"]             = has_hpi

    except Exception as e:
        errors.append((
            row.get("subject_ID", np.nan),
            row.get("group_ID",   np.nan),
            row.get("session_ID", np.nan),
            str(meg_path),
            f"unreadable MEG FIF file: {e}"))

# Report any failures + log into problems_df
if errors:
    print(f"[WARN] {len(errors)} MEG run(s) could not be read from FIF.")
    for sub, grp, ses, p, msg in errors[:20]:
        print(f"  - sub={sub} grp={grp} ses={ses} path='{p}': {msg}")

    # Log all of them into problems_df
    err_df = pd.DataFrame(
        errors,
        columns=["subject_ID", "group_ID", "session_ID", "MEG_fullpath", "reason"])
    # Move 'reason' to be the first column
    err_df = err_df[["reason", "subject_ID", "group_ID", "session_ID", "MEG_fullpath"]]
    problems_df = pd.concat([problems_df, err_df], ignore_index=True)

    if HARD_STOP:
        raise RuntimeError("Hard fail due to unreadable MEG FIF file(s).")
else:
    print("[OK] Read MEG FIF headers for all rows.")

# Optional: round a few numeric fields for readability (does not affect downstream math)
index["sfreq_Hz"]     = pd.to_numeric(index["sfreq_Hz"], errors="coerce").round(6)
index["duration_sec"] = pd.to_numeric(index["duration_sec"], errors="coerce").round(3)
index["duration_min"] = pd.to_numeric(index["duration_min"], errors="coerce").round(3)
index["highpass_Hz"]  = pd.to_numeric(index["highpass_Hz"], errors="coerce").round(3)
index["lowpass_Hz"]   = pd.to_numeric(index["lowpass_Hz"], errors="coerce").round(3)
index["line_freq_Hz"] = pd.to_numeric(index["line_freq_Hz"], errors="coerce").round(3)

print("\n[INFO] Rounded key frequency/duration fields for catalog readability.")

In [ ]:
# =========================
# MEG post-QC normalization & diagnostics
# - Check sampling frequency consistency
# - Summarize line frequency and HPI coverage
# =========================

SFREQ_TOL_HZ = 1e-3  # 0.001 Hz tolerance for comparing sfreq

# 1) Hard check: sfreq_Hz present everywhere
sfreq_series = pd.to_numeric(index["sfreq_Hz"], errors="coerce")
if sfreq_series.isna().any():
    missing_rows = index.loc[
        sfreq_series.isna(),
        ["subject_ID", "group_ID", "session_ID", "MEG_fullpath"]]
    print("[ERR] Some sfreq_Hz values are missing:")
    print(missing_rows.head(20).to_string(index=False))

    # Log to problems_df
    err_df = missing_rows.copy()
    err_df.insert(0, "reason", "missing sampling frequency")
    problems_df = pd.concat([problems_df, err_df], ignore_index=True)

    raise RuntimeError("Hard fail: missing sfreq_Hz values for some MEG runs.")

# 2) Sampling frequency uniformity (diagnostic + optional dropping)
sfreq_round6 = sfreq_series.round(6)
unique_sf = np.sort(sfreq_round6.dropna().unique())

if len(unique_sf) != 1:
    vc = sfreq_round6.value_counts().sort_index()
    majority_sf = vc.idxmax()
    deviation_mask = (sfreq_round6 - majority_sf).abs() > SFREQ_TOL_HZ
    n_dev = int(deviation_mask.sum())

    print("[ERR] sfreq_Hz is not uniform across dataset (rounded to 6 decimals).")
    print(f"      Majority sfreq: {majority_sf} Hz; Deviations: {n_dev} run(s).")

    if n_dev <= 10:
        cols_show = ["subject_ID", "group_ID", "session_ID", "MEG_fullpath", "sfreq_Hz"]
        print("[DETAIL] Deviating rows (up to 10):")
        print(index.loc[deviation_mask, cols_show].to_string(index=False))
    else:
        print("[DETAIL] sfreq_Hz value_counts (rounded to 6 decimals):")
        print(vc.to_string())

    # ----- LOGIC WITH CONFIG-DRIVEN BEHAVIOR + LOGGING -----
    if FORCE_SAME_SAMPLING_FREQUENCY:
        # Log all deviating runs before failing
        bad_rows = index.loc[deviation_mask,
                             ["subject_ID", "group_ID", "session_ID", "MEG_fullpath"]]
        err_df = bad_rows.copy()
        err_df.insert(0, "reason",
                      "non-uniform sampling frequency (force_same_sampling_freq=True)")
        problems_df = pd.concat([problems_df, err_df], ignore_index=True)

        raise RuntimeError(
            "Hard fail due to non-uniform sampling frequencies "
            "(config: force_same_sampling_freq=True).")
    else:
        bad_rows = index.loc[deviation_mask,
                             ["subject_ID", "group_ID", "session_ID", "MEG_fullpath"]]

        if DISCARD_SFREQ_MISMATCHED:
            kept_before = len(index)

            # Log deviating runs that are about to be dropped
            err_df = bad_rows.copy()
            err_df.insert(0, "reason", "non-majority sampling frequency (dropped)")
            problems_df = pd.concat([problems_df, err_df], ignore_index=True)

            index = index.loc[~deviation_mask].reset_index(drop=True)
            print(
                f"[WARN] force_same_sampling_freq=False and discard_sfreq_mismatch=True "
                f"→ dropped {n_dev} run(s) with non-majority sfreq; kept {len(index)} of {kept_before}.")
        else:
            # Log deviating runs that are kept
            err_df = bad_rows.copy()
            err_df.insert(0, "reason", "non-majority sampling frequency (kept)")
            problems_df = pd.concat([problems_df, err_df], ignore_index=True)

            print(
                "[WARN] force_same_sampling_freq=False and discard_sfreq_mismatch=False "
                "→ proceeding with mixed sampling frequencies.\n"
                "       NOTE: You should either resample these runs or treat them separately "
                "before any group-level analysis or source estimation.\n")
else:
    print(f"[OK] sfreq_Hz uniform across all runs: {unique_sf[0]} Hz")

# 3) Basic duration sanity (non-negative, non-zero)
dur = pd.to_numeric(index["duration_sec"], errors="coerce")
bad_dur_mask = (dur <= 0) | dur.isna()
n_bad_dur = int(bad_dur_mask.sum())
if n_bad_dur > 0:
    print(f"[ERR] {n_bad_dur} run(s) have non-positive or missing duration_sec.")
    examples = index.loc[
        bad_dur_mask,
        ["subject_ID", "group_ID", "session_ID", "MEG_fullpath", "duration_sec"]].head(12)
    print(examples.to_string(index=False))

    # Log all bad-duration runs
    bad_rows = index.loc[bad_dur_mask,
                         ["subject_ID", "group_ID", "session_ID", "MEG_fullpath"]]
    err_df = bad_rows.copy()
    err_df.insert(0, "reason", "invalid duration_sec (<=0 or NaN)")
    problems_df = pd.concat([problems_df, err_df], ignore_index=True)

    raise RuntimeError("Hard fail: invalid duration_sec values for some MEG runs.")
else:
    print("[OK] All MEG runs have positive duration_sec values.")

# 4) Line frequency summary
line_freq = pd.to_numeric(index["line_freq_Hz"], errors="coerce")
print("\n[INFO] line_freq_Hz distribution (including NaNs):")
print(line_freq.value_counts(dropna=False).to_string())

if line_freq.isna().any():
    n_missing_lf = int(line_freq.isna().sum())
    print(f"[WARN] {n_missing_lf} run(s) have missing line_freq_Hz in the header.")
else:
    print("[OK] line_freq_Hz present for all runs.")

# 5) HPI coverage summary
if "has_HPI" in index.columns:
    has_hpi_series = index["has_HPI"].astype("boolean")
    print("\n[INFO] HPI coverage (has_HPI):")
    print(has_hpi_series.value_counts(dropna=False).to_string())
    n_no_hpi = int((has_hpi_series == False).sum())
    if n_no_hpi > 0:
        print(f"[WARN] {n_no_hpi} run(s) appear to lack HPI information.")
else:
    print("\n[INFO] 'has_HPI' column not present; skipping HPI coverage summary.")

Extra data-filtering step: REMOVE (or strongly warn about) any files lacking HPI info:

In [ ]:
# =========================
# Drop / warn about runs lacking HPI information
# =========================

if "has_HPI" not in index.columns:
    print("[WARN] 'has_HPI' column not present — cannot evaluate HPI coverage.")
else:
    hpi_series = index["has_HPI"].astype("boolean")
    missing_mask = (hpi_series == False) | (hpi_series.isna())
    n_missing = int(missing_mask.sum())

    if n_missing == 0:
        print("[OK] All MEG runs contain HPI information.")
    else:
        print(f"[WARN] {n_missing} MEG run(s) lack HPI information.")
        bad_rows = index.loc[
            missing_mask,
            ["subject_ID", "group_ID", "session_ID", "MEG_fullpath"]]

        print("[DETAIL] Example missing-HPI rows (up to 20):")
        print(bad_rows.head(20).to_string(index=False))

        # Log all missing-HPI runs (reason depends on whether we drop them)
        if DROP_MISSING_HPI:
            err_df = bad_rows.copy()
            err_df.insert(0, "reason", "missing HPI info (dropped)")
            problems_df = pd.concat([problems_df, err_df], ignore_index=True)

            before = len(index)
            index = index.loc[~missing_mask].reset_index(drop=True)
            print(f"[INFO] drop_missing_HPI_info=True → dropped {n_missing} rows.")
            print(f"[INFO] Remaining runs: {len(index)} (from original {before}).")
        else:
            err_df = bad_rows.copy()
            err_df.insert(0, "reason", "missing HPI info (kept)")
            problems_df = pd.concat([problems_df, err_df], ignore_index=True)

            print("\n[WARN] drop_missing_HPI_info=False → keeping missing-HPI runs.")
            print("       These runs *will* fail later stages unless HPI data can be manually recovered!!!\n")

Assign "cluster labels" to different scanning protocols (as inferred from available acquisition parameter data):

In [ ]:
# =========================
# MEG protocol clustering (revised, less-fragmented)
# Assigns compact alphabetic protocol_code labels (A, B, C, ...)
# =========================

# ---- TOLERANCE KNOBS (can later be promoted to config.yaml if desired)
SFREQ_DECIMALS     = 3   # sfreq_Hz rounding in Hz
LINE_FREQ_DECIMALS = 1   # line_freq_Hz rounding in Hz

n_rows = len(index)

# 1) Build canonical parameter view used for clustering
#    (keep this local in `canon` so we don't clutter `index`)
canon = pd.DataFrame(index=index.index)

# Sampling frequency (rounded)
if "sfreq_Hz" in index.columns:
    canon["sfreq_Hz"] = (
        pd.to_numeric(index["sfreq_Hz"], errors="coerce")
        .round(SFREQ_DECIMALS))
else:
    canon["sfreq_Hz"] = np.nan

# Line frequency (rounded)
if "line_freq_Hz" in index.columns:
    canon["line_freq_Hz"] = (
        pd.to_numeric(index["line_freq_Hz"], errors="coerce")
        .round(LINE_FREQ_DECIMALS))
else:
    canon["line_freq_Hz"] = np.nan

# HPI presence (boolean)
if "has_HPI" in index.columns:
    canon["has_HPI"] = index["has_HPI"].astype("boolean")
else:
    canon["has_HPI"] = pd.Series([np.nan] * n_rows, index=index.index)

# Channel-type presence flags (presence/absence, not exact counts)
def _presence_flag(col_name: str) -> pd.Series:
    if col_name in index.columns:
        vals = pd.to_numeric(index[col_name], errors="coerce")
        return vals.fillna(0) > 0
    else:
        return pd.Series([np.nan] * n_rows, index=index.index)

canon["has_MEG_channels"]  = _presence_flag("n_meg_channels")
canon["has_REF_channels"]  = _presence_flag("n_ref_channels")
canon["has_EEG_channels"]  = _presence_flag("n_eeg_channels")
canon["has_STIM_channels"] = _presence_flag("n_stim_channels")
canon["has_MISC_channels"] = _presence_flag("n_misc_channels")

# If you ever add manufacturer/model columns to `index`, you can optionally include them here
# for opt_key in ["device_type", "device_model", "manufacturer", "system_name"]:
#     if opt_key in index.columns:
#         canon[opt_key] = index[opt_key].astype(str)

# 2) Build the clustering signature from canonicalized values
signature_keys = [
    "sfreq_Hz",
    "line_freq_Hz",
    "has_HPI",
    "has_MEG_channels",
    "has_REF_channels",
    "has_EEG_channels",
    "has_STIM_channels",
    "has_MISC_channels"]

canon_filled = canon[signature_keys].fillna("NA").astype(str)
index["scanner_signature"] = canon_filled.apply(
    lambda row: "|".join(row.values.tolist()),
    axis=1)

# 3) Assign compact alphabetic labels (A, B, ..., Z, AA, AB, ...)
unique_sigs = index["scanner_signature"].drop_duplicates().reset_index(drop=True)
n_sigs = len(unique_sigs)

labels = []
for i in range(n_sigs):
    x = i
    s = ""
    # base-26 "Excel-style" lettering: A..Z, AA..AZ, BA.. etc.
    while True:
        s = chr(65 + (x % 26)) + s
        x = x // 26 - 1
        if x < 0:
            break
    labels.append(s)

sig_to_label = dict(zip(unique_sigs, labels))
index["protocol_code"] = index["scanner_signature"].map(sig_to_label)

# 4) Canonical summary per cluster
cluster_summary = canon.copy()
cluster_summary["protocol_code"] = index["protocol_code"]

summary_cols = ["protocol_code"] + signature_keys
cluster_summary = (
    cluster_summary
    .drop_duplicates()
    .sort_values(["protocol_code"])
    [summary_cols])

print(f"[INFO] Assigned {len(sig_to_label)} unique protocol_code labels (tolerance-aware).")
print(cluster_summary.to_string(index=False))

# 5) Clean up temporary signature
index = index.drop(columns=["scanner_signature"])

-------
#### Final save / export:

In [ ]:
# =========================
# Export warning/problem rows (optional) and main MEG parameter catalog
# =========================

if 'EXPORT_WARNING_CSV' in globals() and EXPORT_WARNING_CSV:
    if not problems_df.empty:
        # Prefer a consistent, MEG-specific name
        out_path = Path(ROOT_DIR) / "MEG_parameter_issues.csv"

        # Nice column ordering: reason first, then core IDs, then anything extra
        preferred_cols = ["reason", "subject_ID", "group_ID", "session_ID", "MEG_fullpath"]
        existing_pref = [c for c in preferred_cols if c in problems_df.columns]
        extra_cols = [c for c in problems_df.columns if c not in existing_pref]

        export_df = problems_df[existing_pref + extra_cols].drop_duplicates()

        export_df.to_csv(out_path, index=False)
        print(f"\n[SAVED] Incomplete/flagged MEG metadata CSV: {out_path} (rows={len(export_df)})")
    else:
        print("\n[INFO] EXPORT_WARNING_CSV is True but no problems were logged → "
              "not writing MEG_parameter_issues.csv.")
else:
    print("\n[INFO] EXPORT_WARNING_CSV is False → not writing MEG_params_incomplete.csv")

# Always save the main parameter index:
index.to_csv(PARAMETER_INDEX_FILEPATH, index=False)
print(f"[SAVED] Main MEG parameter catalog: {PARAMETER_INDEX_FILEPATH}")